In [1]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from omegaconf import OmegaConf
from functools import partial

import numpy as np
import torch
from torchtune import config
from torchtune.data import padded_collate_packed
from torch.utils.data import DataLoader
import torch.nn.functional as F

from dataset_classes import learning_levels_pfa_dataset, PackedOnTheFlyDataset
from evaluation.pfa_evaluation import generate_per_token_losses, decode_token_by_token
from training import SelfPredictionTrainingRecipeDistributed

cfg = OmegaConf.load(f'{PROJECT_ROOT}/configs/llama_0.1B_PHi.yaml')

/home/woody/iwbi/iwbi106h/software/private/conda/envs/hsp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cfg_tokenizer = cfg.tokenizer
tokenizer = config.instantiate(cfg.tokenizer)

In [3]:
included_levels = [0,2]

cfg_dataset = cfg.dataset
cfg_dataset.included_learning_levels = included_levels
packed_on_the_fly = cfg_dataset.pop("packed_on_the_fly", False)
packed_sequence_length = cfg_dataset.pop("packed_sequence_length", 2048)
split_across_pack = cfg_dataset.pop("split_across_pack", False)
num_workers = cfg_dataset.pop("num_workers", 8)

ds = config.instantiate(cfg_dataset, tokenizer)
sample_ds = next(iter(ds))

In [4]:
sample_ds.keys()

dict_keys(['tokens', 'labels', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', '_transition_probs', '_transition_symbols'])

In [5]:
base_dir = '/home/woody/iwbi/iwbi106h/suuraj/models/self-prediction-models'
model_path = 'learning_levels_sweep_x1bmwa36'

base_model_path = os.path.join(base_dir, model_path)
cfg = OmegaConf.load(os.path.join(base_model_path, 'config.yaml'))
cfg.checkpointer.checkpoint_dir = base_model_path
cfg.checkpointer.checkpoint_files = ["torchtune_model_last.pt"]
cfg.train_from_scratch = False
cfg.metric_logger.mode = 'disabled'

recipe = SelfPredictionTrainingRecipeDistributed(cfg=cfg)
recipe.setup(cfg=cfg)

DEBUG:torchtune.utils._logging:Setting manual seed to local seed 2340060461. Local seed is seed + rank = 2340060461 + 0
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
INFO:torchtune.utils._logging:Model is initialized with precision torch.bfloat16.
INFO:torchtune.utils._logging:Memory stats after model init:
	GPU peak memory allocation: 0.18 GiB
	GPU peak memory reserved: 0.20 GiB
	GPU peak memory active: 0.18 GiB
INFO:torchtune.utils._logging:Optimizer is initialized.
INFO:torchtune.utils._logging:information bottleneck: continuous
INFO:torchtune.utils._logging:phi loss factor: 0.001
INFO:torchtune.utils._logging:self critic loss factor: 0.1
INFO:torchtune.utils._logging:Loss is initialized.


run id:  cfr04jaq
None


INFO:torchtune.utils._logging:Dataset and Sampler are initialized.
INFO:torchtune.utils._logging:Learning rate scheduler is initialized.
INFO:torchtune.utils._logging: Profiler config after instantiation: {'enabled': False}


In [6]:
@torch.no_grad()
def process_data(
    recipe,
    num_datapoints=10,
    dataset=None,
    batch_size=4,
):
    """
    Processes data from a dataset to generate a specified number of datapoints
    with per-token loss information.

    This function iterates through the dataset, forms batches, calculates
    per-token losses using the `generate_per_token_losses` function,
    adjusts losses related to hidden state predictions by padding,
    and then splits the batch results into individual datapoint dictionaries.

    Args:
        recipe: A recipe object containing model, tokenizer, dataloader (for default dataset), etc.
        num_datapoints (int, optional): The target number of datapoints to generate. Defaults to 10.
        dataset (Dataset, optional): The dataset to process. If None, uses `recipe._dataloader.dataset`.
                                     Defaults to None.
        batch_size (int, optional): The number of samples to process in each batch. Defaults to 4.

    Returns:
        list: A list of dictionaries, where each dictionary represents a datapoint
              containing tokens, decoded tokens, various per-token losses, and other
              relevant information from the original sample, all as numpy arrays.
    """
    if dataset is None:
        dataset = recipe._dataloader.dataset

    packed_dataset = PackedOnTheFlyDataset(dataset,
                                           max_seq_len=dataset.max_sample_length)
    datapoints = []
    finished_processing_dataset = False

    # Iterate through the dataset in batches.
    for idx in range(0, len(dataset), batch_size):
        print(f"Processing batch {len(datapoints)+1}/{num_datapoints}")
        current_batch_samples = []
        for i in range(batch_size):
            # get next item from the packed_dataset
            try:
                sample = next(packed_dataset)
                current_batch_samples.append(sample)
            except StopIteration:
                finished_processing_dataset = True
                break
        model_batch = padded_collate_packed(current_batch_samples)

        # Calculate per-token losses
        per_token_losses = generate_per_token_losses(recipe,
                                                     model_batch)

        for key, value in per_token_losses.items():
            if 'next' in key:
                per_token_losses[key] = torch.cat([torch.zeros_like(value[:, 0:1]), value], dim=1)
            if key == 'next_token_losses':
                per_token_losses[key] = per_token_losses[key][:, :-1]


        # split into datapoints
        current_datapoints = []
        for b in range(len(current_batch_samples)):
            start_idx = 0
            sample = current_batch_samples[b]
            for i, seq_len in enumerate(sample['seq_lens']):
                end_idx = start_idx + seq_len
                current_datapoint = {}
                for key, value in sample.items():
                    if len(value) != len(sample['tokens']):
                        continue
                    current_datapoint[key] = value[start_idx:end_idx].detach().cpu().numpy()
                for key, value in per_token_losses.items():
                    if key == 'tokens':
                        continue
                    if type(value) != torch.Tensor or value.numel() <= 1:
                        continue
                    value = value[b][start_idx:end_idx]
                    if type(value) is torch.Tensor:
                        value = value.detach().cpu().float().numpy()
                    current_datapoint[key] = value

                # valid datapoint only if not all tokens are 0
                is_valid_datapoint = not np.all(current_datapoint['tokens'] == 0)
                if is_valid_datapoint:
                    current_datapoints.append(current_datapoint)
                start_idx = end_idx
        datapoints.extend(current_datapoints)
        if len(datapoints) >= num_datapoints:
            break
        if finished_processing_dataset:
            break
    if len(datapoints) > num_datapoints:
        datapoints = datapoints[:num_datapoints]
    return datapoints


In [7]:
recipe._model.eval()
datapoints = process_data(recipe, 500, ds, 4)

Processing batch 1/500


DEBUG:torchtune.utils._logging:Using flex attention for attention computation since a BlockMask was passed in.


Processing batch 5/500
Processing batch 9/500
Processing batch 13/500
Processing batch 17/500
Processing batch 21/500
Processing batch 25/500
Processing batch 29/500
Processing batch 33/500
Processing batch 37/500
Processing batch 41/500
Processing batch 45/500
Processing batch 49/500
Processing batch 53/500
Processing batch 57/500
Processing batch 61/500
Processing batch 65/500
Processing batch 69/500
Processing batch 73/500
Processing batch 77/500
Processing batch 81/500
Processing batch 85/500
Processing batch 89/500
Processing batch 93/500
Processing batch 97/500
Processing batch 101/500
Processing batch 105/500
Processing batch 109/500
Processing batch 113/500
Processing batch 117/500
Processing batch 121/500
Processing batch 125/500
Processing batch 129/500
Processing batch 133/500
Processing batch 137/500
Processing batch 141/500
Processing batch 145/500
Processing batch 149/500
Processing batch 153/500
Processing batch 157/500
Processing batch 161/500
Processing batch 165/500
P

In [8]:
datapoints[0].keys()

dict_keys(['tokens', 'labels', 'input_pos', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', 'next_token_losses', 'latent_losses', 'latent_entropy', 'phi_losses'])

In [9]:
def norm_entropy_for_data(datapoints, num):
    entropy = torch.from_numpy(datapoints['latent_entropy'][1:num+1]) 
    normed_entropy = normalize(entropy)
    return datapoints['tokens'] , normed_entropy

def normalize(data):
    min_data = data.min()
    max_data = data.max()
    out = (data-min_data) / (max_data - min_data)
    return out

In [10]:
decoded_tokens = decode_token_by_token(datapoints[0]['tokens'], tokenizer)

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm
import matplotlib.colors
from IPython.display import display, HTML

def colorize_chars(chars, color_array):
    """
    Colorizes a list of single-character strings with forced wrapping for long sequences.

    Args:
        chars (list): A list of single-character strings (tokens).
        color_array (np.ndarray or list): An array of numbers between 0 and 1.

    Returns:
        str: An HTML string with each character colorized, designed to wrap.
    """
    chars = decode_token_by_token(chars, tokenizer)
    cmap = matplotlib.cm.get_cmap('RdBu')
    
    # 1. Define the CSS style for the wrapper container.
    #    The key is 'white-space: normal;' to allow the inline-blocks to wrap.
    wrapper_style = "white-space: normal; line-height: 1.5; border: 1px solid #ddd; padding: 5px;"
    
    # 2. Define the template for the individual token.
    template = '<span class="token-barcode" style="color: #333; background-color: {}; display: inline-block; padding: 2px 1px; margin: 0;">{}</span>'
    
    colored_string = ''
    for char, color in zip(chars, color_array):
        color_hex = matplotlib.colors.rgb2hex(cmap(color)[:3])
        
        # Handle space and control characters
        display_char = '&nbsp' if char == ' ' else char
        if len(char) == 1 and char < ' ':
            display_char = repr(char).strip("'")
            
        colored_string += template.format(color_hex, display_char)
        
    # 3. Wrap the entire colored string in a <div> with the wrapping style.
    final_html = f'<div style="{wrapper_style}">{colored_string}</div>'
    
    return final_html

/tmp/1464362.tinygpu/ipykernel_1835336/1961312327.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = matplotlib.cm.get_cmap('RdBu')


## Sequence learning level

In [19]:
string_len = 1000
level_chars = colorize_chars(datapoints[0]['tokens'][:string_len], normalize(datapoints[0]['learning_level']))
display(HTML(level_chars))

## Sequence entropy

In [20]:
entropy_chars = colorize_chars(*norm_entropy_for_data(datapoints[0], string_len))
display(HTML(entropy_chars))

## Sequence PHi loss

In [23]:
phi_loss_chars = colorize_chars(datapoints[0]['tokens'][:string_len], normalize(datapoints[0]['phi_losses']))
display(HTML(phi_loss_chars))